# PatchTST Energy Forecasting

This notebook trains a PatchTST-style deep learning model on `cleaned_energy_data_model.csv`.

It does all required data preparation before training:

- chronological split only
- train-only outlier handling
- train-only standardization
- leakage-safe calendar covariates
- sliding windows for multi-step forecasting
- train/validation/test metrics
- final next-24-hour forecast

PatchTST works well on Kaggle GPU because it is a PyTorch model.

In [ ]:
import os
import glob
import math
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 160)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

## 1. Load only the main CSV

In [ ]:
DATA_PATH = os.environ.get("DATA_PATH", "")
candidate_paths = [
    DATA_PATH,
    "/kaggle/input/cleaned-energy-data-model/cleaned_energy_data_model.csv",
    "/kaggle/input/energy-consumption/cleaned_energy_data_model.csv",
    r"C:\Users\Chavda\Downloads\cleaned_energy_data_model.csv",
]

if not DATA_PATH:
    candidate_paths.extend(glob.glob("/kaggle/input/**/cleaned_energy_data_model.csv", recursive=True))
    candidate_paths.extend(glob.glob("/kaggle/input/**/*.csv", recursive=True))

DATA_PATH = next((p for p in candidate_paths if p and os.path.exists(p)), None)
if DATA_PATH is None:
    raise FileNotFoundError("Could not find cleaned_energy_data_model.csv. Set DATA_PATH manually.")

raw = pd.read_csv(DATA_PATH)
print(DATA_PATH)
print(raw.shape)
display(raw.head())

In [ ]:
TIME_COL = "start_time"
TARGET_COL = "consumption"
FREQ = "h"

INPUT_LEN = 168       # one week of hourly history
FORECAST_HORIZON = 24 # next 24 hours
BATCH_SIZE = 128
MAX_EPOCHS = 40
PATIENCE = 7

df = raw.copy()
df[TIME_COL] = pd.to_datetime(df[TIME_COL], errors="coerce")
df = df.dropna(subset=[TIME_COL, TARGET_COL]).sort_values(TIME_COL)
df = df.drop_duplicates(subset=[TIME_COL], keep="last")
df = df.rename(columns={TIME_COL: "ds", TARGET_COL: "y"})[["ds", "y"]]

# Ensure hourly continuity. Forward-fill uses only past values.
full_index = pd.date_range(df["ds"].min(), df["ds"].max(), freq=FREQ)
df = df.set_index("ds").reindex(full_index).rename_axis("ds").reset_index()
df["y"] = df["y"].ffill()
df = df.dropna(subset=["y"]).reset_index(drop=True)

print(df.shape)
print(df["ds"].min(), "to", df["ds"].max())
display(df.head())
display(df.tail())

## 2. Metrics

In [ ]:
def metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = y_true != 0
    err = y_true - y_pred
    abs_err = np.abs(err)
    mse = np.mean(err ** 2)
    denom = np.sum(np.abs(y_true))
    ss_res = np.sum(err ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    return {
        "MAE": float(np.mean(abs_err)),
        "MedianAE": float(np.median(abs_err)),
        "MSE": float(mse),
        "RMSE": float(np.sqrt(mse)),
        "MAPE": float(np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100),
        "sMAPE": float(np.mean(2 * abs_err / (np.abs(y_true) + np.abs(y_pred))) * 100),
        "WAPE": float(np.sum(abs_err) / denom * 100) if denom else np.nan,
        "R2": float(1 - ss_res / ss_tot) if ss_tot else np.nan,
        "Accuracy_from_MAPE": float(100 - np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100),
    }

## 3. Chronological train / validation / test split

In [ ]:
train_end = int(len(df) * 0.65)
val_end = int(len(df) * 0.85)

train_df = df.iloc[:train_end].copy()
val_df = df.iloc[train_end:val_end].copy()
test_df = df.iloc[val_end:].copy()

split_summary = pd.DataFrame({
    "split": ["train", "validation", "test"],
    "rows": [len(train_df), len(val_df), len(test_df)],
    "start": [train_df["ds"].min(), val_df["ds"].min(), test_df["ds"].min()],
    "end": [train_df["ds"].max(), val_df["ds"].max(), test_df["ds"].max()],
})
display(split_summary)

## 4. Train-only outlier handling

For PatchTST, we replace training outliers with a robust seasonal median. Validation/test targets are kept original for honest metrics.

In [ ]:
class RobustOutlierHandler:
    def __init__(self, group_cols=("month", "hour"), z_limit=5.0, min_group_size=30):
        self.group_cols = list(group_cols)
        self.z_limit = z_limit
        self.min_group_size = min_group_size
        self.global_median_ = None
        self.global_mad_ = None
        self.group_stats_ = None

    @staticmethod
    def _calendar(frame):
        out = frame[["ds", "y"]].copy()
        out["hour"] = out["ds"].dt.hour
        out["day_of_week"] = out["ds"].dt.dayofweek
        out["month"] = out["ds"].dt.month
        return out

    @staticmethod
    def _mad(values):
        values = np.asarray(values, dtype=float)
        median = np.nanmedian(values)
        mad = np.nanmedian(np.abs(values - median))
        return float(median), max(float(mad), 1e-9)

    def fit(self, train_df):
        tmp = self._calendar(train_df)
        self.global_median_, self.global_mad_ = self._mad(tmp["y"].values)
        rows = []
        for keys, grp in tmp.groupby(self.group_cols):
            if len(grp) < self.min_group_size:
                continue
            median, mad = self._mad(grp["y"].values)
            if not isinstance(keys, tuple):
                keys = (keys,)
            rows.append(dict(zip(self.group_cols, keys), median=median, mad=mad))
        self.group_stats_ = pd.DataFrame(rows)
        return self

    def flag(self, frame):
        tmp = self._calendar(frame)
        tmp = tmp.merge(self.group_stats_, on=self.group_cols, how="left")
        tmp["median"] = tmp["median"].fillna(self.global_median_)
        tmp["mad"] = tmp["mad"].fillna(self.global_mad_).clip(lower=1e-9)
        tmp["robust_z"] = 0.6745 * (tmp["y"] - tmp["median"]) / tmp["mad"]
        tmp["is_impossible"] = tmp["y"] <= 0
        tmp["is_statistical_outlier"] = tmp["robust_z"].abs() > self.z_limit
        tmp["is_outlier"] = tmp["is_impossible"] | tmp["is_statistical_outlier"]
        return tmp[["ds", "median", "robust_z", "is_outlier"]]

    def replace_training_outliers(self, train_df):
        flags = self.flag(train_df)
        cleaned = train_df.copy().merge(flags, on="ds", how="left")
        cleaned["y_original"] = cleaned["y"]
        cleaned.loc[cleaned["is_outlier"].fillna(False), "y"] = cleaned.loc[cleaned["is_outlier"].fillna(False), "median"]
        return cleaned[["ds", "y", "y_original", "is_outlier", "robust_z"]]


outlier_handler = RobustOutlierHandler(z_limit=5.0).fit(train_df)
train_clean = outlier_handler.replace_training_outliers(train_df)
print(f"Training outliers replaced: {int(train_clean['is_outlier'].sum())} / {len(train_clean)}")
display(train_clean.loc[train_clean["is_outlier"].fillna(False)].head())

## 5. Calendar covariates and train-only standardization

In [ ]:
def add_time_features(frame):
    out = frame.copy()
    out["hour"] = out["ds"].dt.hour
    out["day_of_week"] = out["ds"].dt.dayofweek
    out["month"] = out["ds"].dt.month
    out["day_of_year"] = out["ds"].dt.dayofyear
    out["hour_of_week"] = out["day_of_week"] * 24 + out["hour"]
    out["is_weekend"] = out["day_of_week"].isin([5, 6]).astype(float)
    out["is_working_hour"] = out["hour"].between(8, 18).astype(float)
    out["is_peak_morning"] = out["hour"].between(7, 10).astype(float)
    out["is_peak_evening"] = out["hour"].between(17, 21).astype(float)
    out["is_night"] = out["hour"].isin([0, 1, 2, 3, 4, 5]).astype(float)
    out["is_month_start"] = out["ds"].dt.is_month_start.astype(float)
    out["is_month_end"] = out["ds"].dt.is_month_end.astype(float)
    out["season"] = ((out["month"] % 12) // 3).astype(float)
    out["hour_sin"] = np.sin(2 * np.pi * out["hour"] / 24)
    out["hour_cos"] = np.cos(2 * np.pi * out["hour"] / 24)
    out["dow_sin"] = np.sin(2 * np.pi * out["day_of_week"] / 7)
    out["dow_cos"] = np.cos(2 * np.pi * out["day_of_week"] / 7)
    out["month_sin"] = np.sin(2 * np.pi * out["month"] / 12)
    out["month_cos"] = np.cos(2 * np.pi * out["month"] / 12)
    out["doy_sin"] = np.sin(2 * np.pi * out["day_of_year"] / 366)
    out["doy_cos"] = np.cos(2 * np.pi * out["day_of_year"] / 366)
    out["how_sin"] = np.sin(2 * np.pi * out["hour_of_week"] / 168)
    out["how_cos"] = np.cos(2 * np.pi * out["hour_of_week"] / 168)
    return out


model_df = df.copy()
model_df.loc[train_clean.index, "y_clean"] = train_clean["y"].values
model_df.loc[model_df["y_clean"].isna(), "y_clean"] = model_df.loc[model_df["y_clean"].isna(), "y"]
model_df = add_time_features(model_df)

covariate_cols = [
    "is_weekend", "is_working_hour", "is_peak_morning", "is_peak_evening", "is_night",
    "is_month_start", "is_month_end", "season",
    "hour_sin", "hour_cos", "dow_sin", "dow_cos", "month_sin", "month_cos",
    "doy_sin", "doy_cos", "how_sin", "how_cos",
]

# Fit standardization on the training window only.
y_mean = model_df.iloc[:train_end]["y_clean"].mean()
y_std = model_df.iloc[:train_end]["y_clean"].std()
if y_std == 0:
    y_std = 1.0

model_df["y_scaled"] = (model_df["y_clean"] - y_mean) / y_std
model_df["y_true_scaled"] = (model_df["y"] - y_mean) / y_std

cov_mean = model_df.iloc[:train_end][covariate_cols].mean()
cov_std = model_df.iloc[:train_end][covariate_cols].std().replace(0, 1.0)
for c in covariate_cols:
    model_df[c + "_scaled"] = (model_df[c] - cov_mean[c]) / cov_std[c]

scaled_covariate_cols = [c + "_scaled" for c in covariate_cols]
feature_cols = ["y_scaled"] + scaled_covariate_cols

print("target mean/std:", y_mean, y_std)
print("feature count:", len(feature_cols))
display(model_df[["ds", "y", "y_clean", "y_scaled"] + scaled_covariate_cols[:5]].head())

## 6. Sliding windows

Each sample uses `INPUT_LEN=168` hours of past history and predicts the next `FORECAST_HORIZON=24` hours.

The input contains past scaled target plus known calendar covariates. The target is only future scaled consumption.

The validation/test metrics below are rolling-origin metrics: at each forecast origin, the model is allowed to use observed history up to that origin. This matches repeated operational forecasting. For a single blind competition forecast, use only the final-origin forecast section.

In [ ]:
class WindowDataset(Dataset):
    def __init__(self, frame, feature_cols, input_len=168, horizon=24, start_idx=0, end_idx=None):
        self.frame = frame.reset_index(drop=True)
        self.feature_cols = feature_cols
        self.input_len = input_len
        self.horizon = horizon
        if end_idx is None:
            end_idx = len(frame) - 1

        self.X_values = self.frame[feature_cols].values.astype(np.float32)
        self.y_values = self.frame["y_true_scaled"].values.astype(np.float32)
        self.ds_values = self.frame["ds"].values

        self.origins = []
        min_origin = max(input_len - 1, start_idx)
        max_origin = min(end_idx - horizon, len(frame) - horizon - 1)
        for origin in range(min_origin, max_origin + 1):
            self.origins.append(origin)

    def __len__(self):
        return len(self.origins)

    def __getitem__(self, idx):
        origin = self.origins[idx]
        x = self.X_values[origin - self.input_len + 1: origin + 1]
        y = self.y_values[origin + 1: origin + 1 + self.horizon]
        return torch.tensor(x), torch.tensor(y), origin


train_ds = WindowDataset(model_df, feature_cols, INPUT_LEN, FORECAST_HORIZON, start_idx=0, end_idx=train_end - 1)
val_ds = WindowDataset(model_df, feature_cols, INPUT_LEN, FORECAST_HORIZON, start_idx=train_end, end_idx=val_end - 1)
test_ds = WindowDataset(model_df, feature_cols, INPUT_LEN, FORECAST_HORIZON, start_idx=val_end, end_idx=len(model_df) - 1)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=False)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

print("windows:", len(train_ds), len(val_ds), len(test_ds))
x0, y0, o0 = train_ds[0]
print("X:", x0.shape, "y:", y0.shape, "origin:", o0)

## 7. PatchTST model

In [ ]:
class PatchTST(nn.Module):
    def __init__(
        self,
        input_len,
        num_features,
        horizon,
        patch_len=24,
        stride=12,
        d_model=128,
        n_heads=8,
        num_layers=3,
        dropout=0.15,
    ):
        super().__init__()
        self.input_len = input_len
        self.num_features = num_features
        self.horizon = horizon
        self.patch_len = patch_len
        self.stride = stride

        self.num_patches = 1 + (input_len - patch_len) // stride
        self.patch_dim = patch_len * num_features

        self.patch_proj = nn.Linear(self.patch_dim, d_model)
        self.pos_embedding = nn.Parameter(torch.zeros(1, self.num_patches, d_model))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_model * 4,
            dropout=dropout,
            batch_first=True,
            activation="gelu",
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Flatten(),
            nn.Linear(self.num_patches * d_model, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, horizon),
        )

        nn.init.normal_(self.pos_embedding, std=0.02)

    def forward(self, x):
        # x: batch, input_len, num_features
        patches = x.unfold(dimension=1, size=self.patch_len, step=self.stride)
        # batch, num_patches, num_features, patch_len -> batch, num_patches, patch_len * num_features
        patches = patches.permute(0, 1, 3, 2).contiguous().view(x.size(0), self.num_patches, self.patch_dim)
        z = self.patch_proj(patches) + self.pos_embedding
        z = self.encoder(z)
        return self.head(z)


model = PatchTST(
    input_len=INPUT_LEN,
    num_features=len(feature_cols),
    horizon=FORECAST_HORIZON,
    patch_len=24,
    stride=12,
    d_model=128,
    n_heads=8,
    num_layers=3,
    dropout=0.15,
).to(device)

print(model)
print("parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))

## 8. Train with early stopping

In [ ]:
def inverse_y(y_scaled):
    return y_scaled * y_std + y_mean


def evaluate_loader(model, loader):
    model.eval()
    losses = []
    all_true = []
    all_pred = []
    all_origins = []
    criterion = nn.SmoothL1Loss()
    with torch.no_grad():
        for xb, yb, origins in loader:
            xb = xb.to(device)
            yb = yb.to(device)
            pred = model(xb)
            loss = criterion(pred, yb)
            losses.append(loss.item() * len(xb))
            all_true.append(yb.cpu().numpy())
            all_pred.append(pred.cpu().numpy())
            all_origins.extend(origins.numpy().tolist())
    y_true_scaled = np.vstack(all_true)
    y_pred_scaled = np.vstack(all_pred)
    y_true = inverse_y(y_true_scaled)
    y_pred = inverse_y(y_pred_scaled)
    return np.sum(losses) / len(loader.dataset), y_true, y_pred, all_origins


criterion = nn.SmoothL1Loss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2)

best_val = float("inf")
best_state = None
bad_epochs = 0
history = []

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    train_loss_sum = 0.0
    for xb, yb, _ in train_loader:
        xb = xb.to(device)
        yb = yb.to(device)
        optimizer.zero_grad()
        pred = model(xb)
        loss = criterion(pred, yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        train_loss_sum += loss.item() * len(xb)

    train_loss = train_loss_sum / len(train_loader.dataset)
    val_loss, val_true, val_pred, _ = evaluate_loader(model, val_loader)
    val_m = metrics(val_true.ravel(), val_pred.ravel())
    scheduler.step(val_loss)

    row = {"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss, "val_MAPE": val_m["MAPE"], "val_RMSE": val_m["RMSE"]}
    history.append(row)
    print(f"epoch {epoch:02d} train_loss={train_loss:.5f} val_loss={val_loss:.5f} val_MAPE={val_m['MAPE']:.3f}%")

    if val_loss < best_val:
        best_val = val_loss
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        bad_epochs = 0
    else:
        bad_epochs += 1
        if bad_epochs >= PATIENCE:
            print("Early stopping")
            break

model.load_state_dict(best_state)
history_df = pd.DataFrame(history)
display(history_df.tail())

## 9. Train / validation / test rolling-origin metrics

In [ ]:
train_loss, train_true, train_pred, train_origins = evaluate_loader(model, train_loader)
val_loss, val_true, val_pred, val_origins = evaluate_loader(model, val_loader)
test_loss, test_true, test_pred, test_origins = evaluate_loader(model, test_loader)

metric_report = pd.DataFrame([
    {"split": "train", **metrics(train_true.ravel(), train_pred.ravel())},
    {"split": "validation", **metrics(val_true.ravel(), val_pred.ravel())},
    {"split": "test", **metrics(test_true.ravel(), test_pred.ravel())},
])
display(metric_report)

gap = metric_report.loc[metric_report["split"].eq("test"), "MAPE"].iloc[0] - metric_report.loc[metric_report["split"].eq("train"), "MAPE"].iloc[0]
print(f"Train-test MAPE gap: {gap:.3f} percentage points")

metric_report.to_csv("patchtst_train_val_test_metrics.csv", index=False)
print("Wrote patchtst_train_val_test_metrics.csv")

## 10. Horizon-wise metrics

In [ ]:
horizon_rows = []
for h in range(FORECAST_HORIZON):
    row = {"horizon": h + 1}
    row.update(metrics(test_true[:, h], test_pred[:, h]))
    horizon_rows.append(row)
horizon_metrics = pd.DataFrame(horizon_rows)
display(horizon_metrics)
horizon_metrics.to_csv("patchtst_horizon_metrics.csv", index=False)
print("Wrote patchtst_horizon_metrics.csv")

## 11. Visual check

In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(history_df["epoch"], history_df["train_loss"], label="train loss")
plt.plot(history_df["epoch"], history_df["val_loss"], label="validation loss")
plt.title("PatchTST training curve")
plt.xlabel("epoch")
plt.ylabel("SmoothL1 loss")
plt.legend()
plt.tight_layout()
plt.show()

example = 0
plt.figure(figsize=(14, 5))
plt.plot(range(1, FORECAST_HORIZON + 1), test_true[example], marker="o", label="actual")
plt.plot(range(1, FORECAST_HORIZON + 1), test_pred[example], marker="o", label="predicted")
plt.title("Example 24-hour PatchTST forecast on test")
plt.xlabel("horizon")
plt.ylabel("MWh")
plt.legend()
plt.tight_layout()
plt.show()

## 12. Final next-24-hour forecast

In [ ]:
model.eval()
last_origin = len(model_df) - 1
x_last = model_df[feature_cols].values.astype(np.float32)[last_origin - INPUT_LEN + 1:last_origin + 1]
with torch.no_grad():
    pred_scaled = model(torch.tensor(x_last).unsqueeze(0).to(device)).cpu().numpy()[0]
pred = inverse_y(pred_scaled)

future_ds = pd.date_range(model_df["ds"].max() + pd.Timedelta(hours=1), periods=FORECAST_HORIZON, freq=FREQ)
forecast = pd.DataFrame({
    "ds": future_ds,
    "horizon": np.arange(1, FORECAST_HORIZON + 1),
    "patchtst_yhat": pred,
})
forecast.to_csv("patchtst_next_24h_forecast.csv", index=False)
display(forecast)
print("Wrote patchtst_next_24h_forecast.csv")

## Notes for reporting

PatchTST differs from Prophet:

- Prophet uses trend/seasonality and known future regressors.
- PatchTST learns directly from the last 168 hours as patches and predicts all 24 future hours together.
- Standardization is fit only on the training period.
- Validation/test targets are untouched, so metrics are honest.